In [ ]:
import os
import json
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore
from pinecone import Pinecone
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate

In [ ]:
load_dotenv()

Generate json with metadata from a pdf document

In [ ]:
import json
import re
from pathlib import Path

from pypdf import PdfReader

pdf_path = Path(
    r"C:\Meher\codebase\stockanalysis\data\stocks_edu_posts.pdf"
)
output_path = pdf_path.with_suffix(".json")

reader = PdfReader(pdf_path)

post_date_pattern = re.compile(
    r"^[A-Z][a-z]{2} \d{1,2}, \d{4}.*https://x\.com/"
)

records = []
last_post_title = None

for page_number, page in enumerate(reader.pages, start=1):
    content = page.extract_text() or ""
    lines = [
        line.strip()
        for line in content.splitlines()
        if line.strip()
    ]

    date_index = next(
        (
            i for i, line in enumerate(lines)
            if post_date_pattern.match(line)
        ),
        None,
    )

    if date_index is not None:
        # This PDF places the category first, then the post heading.
        title = " ".join(lines[1:date_index]).strip()

        # Fix known spacing artifacts in extracted headings.
        for old, new in {
            "T echnical": "Technical",
            "T opping": "Topping",
            "T argets": "Targets",
        }.items():
            title = title.replace(old, new)

        if not title:
            raise ValueError(
                f"No post heading found on page {page_number}"
            )

        last_post_title = title

    elif page_number == 1:
        title = "Collected Threads & Lessons: Overview"

    elif page_number <= 5:
        title = f"Table of Contents - Page {page_number}"

    elif any(re.search(r"\d+ posts$", line) for line in lines):
        # Category divider page.
        heading = " ".join(
            line for line in lines
            if not line.startswith("Venu's Educational Posts")
        )
        title = re.sub(r"\d+ posts$", "", heading).strip()
        last_post_title = None

    elif last_post_title:
        title = f"{last_post_title} (continued)"

    else:
        title = " ".join(lines[:2]) or f"Page {page_number}"

    records.append({
        "content": content,
        "metadata": {
            "doc_type": "educational_posts",
            "source": pdf_path.name,
            "author": "Venu (@Venu_7_)",
            "title": title,
            "page_number": page_number,
            "total_pages": len(reader.pages),
        },
    })

output_path.write_text(
    json.dumps(records, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print(f"Saved {len(records)} page records to: {output_path}")

Load the generated json from pdf to docs

In [ ]:
with open(
    r"C:\Meher\codebase\stockanalysis\data\stocks_edu_posts.json",
    "r",
    encoding="utf-8",
) as f:
    kb = json.load(f)

from langchain_core.documents import Document


docs = [
    Document(page_content=entry["content"], metadata=entry["metadata"])
    for entry in kb
]

In [ ]:
docs[1].page_content[:500]  # Display the first 500 characters of the second document

In [ ]:
def wrap_text(text, width=80):
    return '\n'.join([text[i:i+width] for i in range(0, len(text), width)])

for doc in docs:
    print(wrap_text(doc.page_content))
    print("-"*100)

In [ ]:
# chunk the data

# ref: https://python.langchain.com/docs/concepts/text_splitters/
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,  # chunk size (characters)
    chunk_overlap=100,  # chunk overlap (characters)
    add_start_index=True,  # track index in original document
)

splits = text_splitter.split_documents(docs)

In [ ]:
# See what chunking did: 51 documents become a larger number of smaller chunks.
print(f"Documents in:  {len(docs)}")
print(f"Chunks out:    {len(splits)}")
print("-" * 100)

# Find the longest document and show how it got split (with overlap + start_index).
longest_idx = max(range(len(docs)), key=lambda i: len(docs[i].page_content))
parent = docs[longest_idx]

child_chunks = [
    s for s in splits if s.metadata.get("title") == parent.metadata["title"]
]
print(f"It split into {len(child_chunks)} chunks:\n")
for i, chunk in enumerate(child_chunks):
    start = chunk.metadata["start_index"]
    print(f"[chunk {i} | start_index={start} | {len(chunk.page_content)} chars]")
    print(chunk.page_content)
    print("-" * 100)

In [ ]:
for split in splits:
    print(wrap_text(split.page_content))
    print("-"*100)

In [ ]:
from pinecone import Pinecone
from langchain_pinecone import PineconeEmbeddings, PineconeVectorStore
#Indexing

#define the embeddings model
#ref: https://python.langchain.com/docs/integrations/text_embedding/
#text-embedding-3-small defaults to 1536 dims, but the text-embedding-3-* models
#support shortening the output via `dimensions`. We use 512 to match the index.
embeddings = PineconeEmbeddings(
    model="llama-text-embed-v2",
    pinecone_api_key=os.environ["PINECONE_API_KEY"],
    query_params={
        "input_type": "query",
        "dimension": 512,
    },
    document_params={
        "input_type": "passage",
        "dimension": 512,
    },
)

#connect to an existing Pinecone index
#(create/manage the index in the Pinecone console; its dimension must be 512
# to match the `dimensions=512` embeddings above, with the cosine metric)
#ref: https://python.langchain.com/docs/integrations/vectorstores/pinecone/
pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])
index = pc.Index(os.environ["PINECONE_INDEX_NAME"])

#define the vector store (this notebook's data lives in the "shopeasy-basic-rag" namespace)
#ref: https://python.langchain.com/docs/concepts/vectorstores/
vector_store = PineconeVectorStore(
    index=index,
    embedding=embeddings,
    namespace="stock-analysis-nsp",
)

In [ ]:
import time
from pinecone import ServerlessSpec
from langchain_pinecone import PineconeVectorStore

index_name = "stock-analysis-index"

if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        vector_type="dense",
        dimension=512,
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1",
        ),
    )

while not pc.describe_index(index_name).status["ready"]:
    time.sleep(2)

index = pc.Index(index_name)

vector_store = PineconeVectorStore(
    index=index,
    embedding=embeddings,
    namespace="stock-analysis-nsp",
)

# Stable IDs prevent duplicates when rerunning with the same splits.
ids = [f"stocks-edu-{i}" for i in range(len(splits))]

document_ids = vector_store.add_documents(
    documents=splits,
    ids=ids,
)

print(f"Uploaded {len(document_ids)} documents")

In [ ]:
index.fetch(ids=[document_ids[10]], namespace="stock-analysis-nsp")

In [ ]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_openai import ChatOpenAI

env_path = Path(r"C:\Meher\codebase\stockanalysis\.env")
load_dotenv(env_path)

#configure the llm
llm = ChatOpenAI(
    model=os.environ["NEBIUS_MODEL"],
    api_key=os.environ["NEBIUS_API_KEY"],
    base_url="https://api.tokenfactory.nebius.com/v1/",
    temperature=0,
    use_responses_api=False,
)

#set the prompt template
template = """You are a trading/investment assistant for stock market investment.
Use the following pieces of retrieved internal knowledge-base context to help answer the customer's question.
If the context doesn't contain the answer, you are free to get information from web but please dont guess.
Be concise and practical: state the likely cause and the next step the agent should take.

Context:
{context}

Customer question: {question}

Support guidance:"""

rag_prompt_template = PromptTemplate.from_template(template)

In [ ]:
# A sample customer support ticket (README Query 1).
user_question = "how do we know if stock is undervalued or overvalued?"

In [ ]:
# Now, we'll use the vector store's similarity_search method to find the top 5 most similar chunks to the customer issue.
retrieved_docs = vector_store.similarity_search(user_question, k=10)

In [ ]:
# Review the retrieved chunks. We print the metadata header so we can SEE the noise:
# semantic search will surface relevant trading docs, but may also pull in.
for doc in retrieved_docs:
    m = doc.metadata
    print(f"[{m['doc_type']} / {m['title']}]")
    print(doc.page_content)
    print("-" * 100)

In [ ]:
# The similarity_search_with_score method returns the chunks along with their similarity scores.
vector_store.similarity_search_with_score(user_question, k=5)

In [ ]:
#generate answer

docs_content = "\n\n".join(doc.page_content for doc in retrieved_docs)
prompt = rag_prompt_template.invoke({"question": user_question, "context": docs_content})
response = llm.invoke(prompt)

In [ ]:
response.content

In [ ]:
# We can also include citations in our response by extracting the source from the metadata of the retrieved docs.
# This dataset has no "source" URL, so we cite each doc by its title and doc_type instead.

sources = [f"{doc.metadata['title']} ({doc.metadata['doc_type']})" for doc in retrieved_docs]

print(f"Sources: {sources}\n\n")
print(f'Answer: {response.content}')

In [ ]:
# Now, let's put it all together in a single function.
def generate_answer(user_question):
    #retrieve the relevant docs
    retriever = vector_store.as_retriever(search_kwargs={"k": 5}, search_type='similarity')
    retrieved_docs = retriever.invoke(user_question)

    #generate
    docs_content = "\n\n".join(doc.page_content for doc in retrieved_docs)
    prompt = rag_prompt_template.invoke({"question": user_question, "context": docs_content})
    response = llm.invoke(prompt)

    return response.content

generate_answer(user_question)